# DEVP 209 Section 5

In this lab, we will be using Python and Pandas to understand Bayes' rule of conditional probability and also apply conditionality to use indicator variables

## Imports

In [ ]:
import pandas as pd                 # We can give aliases to packages; can call it whatever I want to save typing
import datetime                     # This allows me to use dates
import numpy as np                  # This is the second most popular package after Pandas most likely
import statsmodels.api as sm        # Used for regressions
import  matplotlib.pyplot as plt    # Plotting library

### Practicing Bayes' rule

First, let us see something about how a dataset is represented

In [ ]:
arrests = pd.read_csv('arrests.csv')
print(len(arrests))
arrests.head()

I have a dataset of removals from our ICE data work (https://deportationdata.org/data/ice.html)
Let us see what happens when I do some of the filtering in Pandas that we have already learned

In [ ]:
(arrests['Departure Country'] == 'MEXICO')

So it seems that if I do a check for if the `'Departure Country'` column is equal to the value of `'MEXICO'` then I get a list of `True` and `False` values. This is simply checking every single row in this column and letting us know whether or not our check is right or wrong. Something interesting though happens when I do the following

IMPORTANT TO NOTE: `True` when used in math is the same as 1 and `False` is the same as 0

In [ ]:
(arrests['Departure Country'] == 'MEXICO').sum()

Here I see that there are 73,037 different removals for people from Mexico. Above, I can also see that there 265,226 different rows. This means that I can simply do a sum here and divide by all of the rows to see the probability of being a Mexican out of the people that are removed. This is shown below

In [ ]:
probabilityOfMexicanArrest = (arrests['Departure Country'] == 'MEXICO').sum()/len(arrests)
print(probabilityOfMexicanArrest)

Another way we can do this is simply taking the mean value. This can be interpreted simply since we have a bunch of `True` values that equal 1, then we are finding the average number of 1s which is the same as the probability

In [ ]:
probabilityOfMexicanArrest2 = (arrests['Departure Country'] == 'MEXICO').mean()
print(probabilityOfMexicanArrest2)

This is our first real introduction on how we can use filtering to find probabilities in data.

Below we are going to go through some exercises to practice using the above information to see Bayes' rule using a new set of data: removals.

It is so long that it actually was split into two lol so we need to combine them as below

In [ ]:
removals = pd.read_csv('removals.csv')
removals.head()

In [ ]:
#Finding the sample space of possible Felon values
removals['Felon'].unique()

Let us now try to answer some questions about our dataset to understand the population
* What is the sample space of Departure Countries?
* What is the Probability of being older than 30?
* What is the number of rows that represent the Union of people from COLOMBIA and those that have had a prior deportation?
* What is the number of rows that represent the Intersection of People born after 2000 that are Female?
* What is the complement to the above question?
* What is the probability that a removed person was `'Not an Aggravated Felon'` OR has nothing listed for the felony charge? Hint: OR in Pandas is `|` and you will need to try `pd.isnull(removals['Felon'])`

Now we will try to answer some questions about probability and use Bayes' rule; first look at the given example

In [ ]:
# Probability male
P_A = (removals['Departure Country'] == 'MEXICO').mean()
P_B = (removals['Gender'] == 'Male').mean()

# Probability male and mexican
P_AB = ((removals['Departure Country'] == 'MEXICO') & (removals['Gender'] == 'Male')).sum()/len(removals)

# Likelihood: P(MEXICO | MALE) = P(MEXICO AND MALE) /P(MALE)
conditional_AB = P_AB / P_B
# Likelihood: P(Male | MEXICO)
conditional_BA = P_AB / P_A

print(conditional_AB)
print(conditional_BA)


This means that 15% of Male removals are to Mexico and 70% of people removed to Mexico are Male!

In [ ]:
len(arrests.loc[arrests['Gender']=='Female'])

In [ ]:
# We can do this again by doing the powerful groupby command we have learned in prior classes
arrests.groupby('Gender')['Departure Country'].value_counts(normalize=True)

# What is happening here is we are counting the total number of people in each country for each gender, then 'normalizing' which means dividing by the total number
# In practice, this means that we have for instance 3721 Females deported from Mexico, but 33794 total females, giving a conditional probability of 3721/33794 = 38.8%

Let us now try to figure out some conditional values in our dataset
* What is probability of a female removed if they are younger than 30?
* What is the probability of being a Venezuelan detained if their Final Program was Border Patrol?
* What is the odds that they had a prior departure if they have Felon value of '`Drugs`'?
* What are the odds that people that are not from Mexico are sent to Mexico?
* How much more/less likely are Males to have a Felon value of `'Drugs'` than females? Challenge: How does this vary by country and which country has the greatest discrepancy?

## Challenge Section: Regressing using indicator variables

Let us now use the ArrestsByState data and determine how OLS changes with the presence of an indicator variable

### Step 0: Import the data

In [ ]:
arrests_state_country = pd.read_csv('ArrestsStateCountry.csv', sep='\t')
arrests_state_country.head()

### Step 1: Create a scatter plot that is not a time series

First step is creating a dataframe that simply relates two numerical values that ~might~ be related

In [ ]:
plt.scatter(arrests_state_country['arrests_perdaybiden'], 
           arrests_state_country['arrests_perdaytrump'])

# we can also add titles as follows
plt.ylabel('Arrests per Day (Trump Administration)')
plt.xlabel('Arrests per Day (Biden Administration)')
plt.title('Arrests per Day Trump vs Biden by State-Country Combination')
plt.show()

### Step 2: Add an indicator variable

As we learned in class, an indicator variable is a column of 1s and 0s. I am making an indicator variable for if a country is in Europe

In [ ]:
# I Googled these
european_countries = ['GERMANY', 'FRANCE', 'ITALY', 'SPAIN', 'PORTUGAL', 'POLAND', 'NETHERLANDS',
                      'BELGIUM', 'SWEDEN', 'NORWAY', 'DENMARK', 'FINLAND', 'ICELAND', 'IRELAND',
                      'GREECE', 'AUSTRIA', 'SWITZERLAND', 'HUNGARY', 'CZECH REPUBLIC', 'SLOVAKIA',
                      'SLOVENIA', 'CROATIA', 'SERBIA', 'ROMANIA', 'BULGARIA', 'ESTONIA', 'LATVIA', 'LITHUANIA', 'UKRAINE', 'RUSSIA', 'TURKEY']

arrests_state_country['European'] = (arrests_state_country['citizenshipcountry'].isin(european_countries)).astype(int)
arrests_state_country.head()


### Step 3: Regress on the indicator variable

In [ ]:
arrests_state_country = arrests_state_country.dropna(subset=['arrests_perdaybiden','arrests_perdaytrump'])

In [ ]:
# Define regression: arrests_perdaytrump ~ arrests_perdaybiden + European
X = arrests_state_country[["arrests_perdaybiden", "European"]]
X = sm.add_constant(X) # This is the same as adding a column of all 1s to the country_stats dataframe called 'intercept' and including that in the above selected columns
y = arrests_state_country['arrests_perdaytrump']

model = sm.OLS(y, X).fit()
print(model.summary())

### Challenge 1: Interpret the indicator variable and the intercept

### Challenge 2: Add an indicator variable on the slope

### Challenge 3: Plot the above using what you had done in section 4